In [1]:
# NOTE：喂给神经网络之前，先把数据归一化到 [0, 1] 或者均值为 0、方差为 1，以适应底层BatchNorm的初始化的均值和方差

import sys,os
from datetime import datetime
from pathlib import Path
import json
import torch # 张量、自动求导等
import torch.nn as nn # 网络层，Conv1D、LSTM等
import torch.nn.functional as F # 函数层， ReLU、cross_entropy等
from pyexpat import features
from torchinfo import summary
from torch.utils.data import DataLoader,TensorDataset
import time
import gc
from sklearn.metrics import roc_auc_score
import re
import math
import mne
from zmq import DEALER

from train import learning_rate, hidden_size
import numpy as np

root = Path(r"C:\Users\hank\Desktop\Research Reproduction\dianxian\self")
chb_data_path = Path(r"D:\data\dianxian\channal_select\chb03")

if str(root) not in sys.path:
    sys.path.append(str(root))

import tools as t

log_dir = root / "train_log"
embeddings = root / 'embeddings'
weight = root / 'weight'
Fp1_F7 = embeddings / 'Fp1-F7'
Fp2_F8 = embeddings / 'Fp2-F8'
T7_P7 = embeddings / 'T7-P7'
T8_P8 = embeddings / 'T8-P8'
P7_O1 = embeddings / 'P7-O1'
P8_O2 = embeddings / 'P8-O2'

seed = 3
val = (0.2,0.1)
method = "LOOCV"
out_channals = 32
batch = 32
elec = 6
embed_dim = 32
num_heads = 2
window_length = 10 * 256
window_overlap = 5 * 256
elec_names = ['Fp1-F7', 'Fp2-F8', 'T7-P7', 'T8-P8', 'P7-O1', 'P8-O2']

In [2]:
from models import Detection,CommonConv2d,BILSTMClassify

C:\Users\hank\Desktop\Research Reproduction\dianxian\self\models.py:259: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage(montage_name)


In [3]:
# m = CommonConv2d()
# summary(m,(32,1,6,2500)) # (32, 32, 6, 78)

In [4]:
from datetime import datetime

In [5]:
windows,meta = t.preprocess(r"D:\data\dianxian\wang2021\chb22",SPH=2*60,SOP=30*60,interictal_thr=2*60*60,window_length=4,window_overlap=2,elec_names=elec_names)

# 原始是伏特，尺度 ~1e-4；标准化放到每折里做，只用该折训练集的统计量


[边界] chb22 末尾不足整窗: ['chb22_11.edf(丢 1.0s)']
[chb22] 原始窗 55774
[chb22]   label=0 39544
[chb22]   label=1 2366
[chb22]   label=2 96
[chb22]   label=3 13768
[chb22]   excluded(overlap) 22
[chb22]   excluded(sph) 174
[chb22]   excluded(exclusion) 13572
[chb22] usable 42006 / excluded 13768
[chb22] Fp1-F7 窗 55774
[chb22] Fp2-F8 窗 55774
[chb22] T7-P7 窗 55774
[chb22] T8-P8 窗 55774
[chb22] P7-O1 窗 55774
[chb22] P8-O2 窗 55774


In [6]:
# TODO: 教师的折要向后看齐到学生的折（用 student_data_set 的索引取 label 0/2 再平衡），
# 两边的发作集合不同（chb15 20 vs 18、chb16 10 vs 9），多出来的教师折不用

teacher_data_set = t.teacher_train_val_test(elec_names, meta, windows, seed, val=(0.1, 0.1), method="LOOCV")
student_data_set = t.train_val_test(elec_names, meta, windows, seed, val=(0.1, 0.2), method="LOOCV")

# 每折标准化参数：除重叠窗、除该折 val/test，teacher 和 student 共用
teacher_stats = [t.fold_norm_stats(meta, windows, elec_names, d) for d in teacher_data_set]

[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       34/34          35/35          27/27          54078/0         
[划分] S1       34/34          27/27          35/35          54070/0         
[划分] S2       27/27          35/35          34/34          50143/0         
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] metho

In [7]:
# 教师每折重训一次（LOOCV），六通道一起喂
teacher_result = {}
optimizer_lr = 0.001
loss_fuction = nn.BCEWithLogitsLoss() # 自带sigmoid()
epochs = 100
patience = 15
now = datetime.now().strftime("%y%m_%d_%H%M%S")

for fold in range(len(teacher_data_set)):
    d = teacher_data_set[fold]

    # 复用该折统计量（已排除重叠窗和 val/test），避免测试集信息回流
    mean,std = teacher_stats[fold]
    teacher_train_dataloader,teacher_val_dataloader,teacher_test_dataloader = [
        DataLoader(TensorDataset(t.norm_wins(d[key+"_wins"],mean,std),d["y_"+key]),
                   batch_size=batch,shuffle=(key=="train"))
        for key in ("train","val","test")]

    teacher = Detection().to("cuda")
    optimizer = torch.optim.Adam(teacher.parameters(),lr=optimizer_lr)
    schedule = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    )

    # 验证集每边只有三四十个窗，AUC 会在 0.5 附近乱跳，按 loss 选模型
    best_val_loss = np.inf
    best_state_dict = {}
    patience_count = 0
    print(f"===== fold {fold+1} seizure {d['seizure_id']} =====")

    for epoch in range(epochs):
        avg_train_loss = 0
        avg_val_loss = 0

        teacher.train()
        for (x,y) in teacher_train_dataloader:
            x,y = x.to("cuda"),y.to("cuda")
            optimizer.zero_grad()
            out = teacher(x)
            loss = loss_fuction(out, y)
            loss.backward()
            optimizer.step()
            avg_train_loss += loss.item()

        all_val_probs = []
        all_val_targets = []
        teacher.eval()
        with torch.no_grad():
            for (x,y) in teacher_val_dataloader:
                x,y = x.to("cuda"),y.to("cuda")
                out = teacher(x)
                loss = loss_fuction(out, y)
                avg_val_loss += loss.item()
                all_val_probs.extend(torch.sigmoid(out).cpu().numpy().reshape(-1))
                all_val_targets.extend(y.cpu().numpy().reshape(-1))

        avg_train_loss /= len(teacher_train_dataloader)
        avg_val_loss /= len(teacher_val_dataloader)
        val_auc = roc_auc_score( all_val_targets,all_val_probs)

        # schedule.step(val_auc)
        schedule.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"epoch:{epoch+1} avg_loss:{avg_train_loss:.4f} avg_val_loss:{avg_val_loss:.4f},val_auc:{val_auc:.4f},lr:{current_lr:.8f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_count = 0
            best_state_dict = {k:v.cpu().clone() for k,v in teacher.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= patience:
                print("Early stop！！！")
                break

    # 用最好的权重测这一折的发作
    teacher.load_state_dict(best_state_dict)
    teacher.eval()
    teacher_targets,teacher_probs = [],[]
    with torch.no_grad():
        for (x,y) in teacher_test_dataloader:
            x,y = x.to("cuda"),y.to("cuda")
            teacher_probs.extend(torch.sigmoid(teacher(x)).cpu().numpy().flatten())
            teacher_targets.extend(y.cpu().numpy().flatten())

    # 必须转成 ndarray 才能做布尔索引，list 会被当成 list[0]
    teacher_targets = np.array(teacher_targets)
    teacher_probs = np.array(teacher_probs)

    teacher_auc = roc_auc_score(teacher_targets,teacher_probs)
    teacher_acc = ((teacher_probs > 0.5) == teacher_targets).mean()
    teacher_result[f"fold_{fold+1}"] = {"seizure_id":int(d['seizure_id']),
                                        "n_test":len(teacher_targets),
                                        "auc":round(float(teacher_auc),4),
                                        "acc":round(float(teacher_acc),4)}

    path = weight / f"{now}_teacher_fold{fold+1}.pth"
    torch.save(best_state_dict,path)

===== fold 1 seizure 0 =====
epoch:1 avg_loss:0.6950 avg_val_loss:0.7037,val_auc:0.1449,lr:0.00100000
epoch:2 avg_loss:0.6651 avg_val_loss:0.6997,val_auc:0.8971,lr:0.00100000
epoch:3 avg_loss:0.6641 avg_val_loss:0.6946,val_auc:0.9159,lr:0.00100000
epoch:4 avg_loss:0.6642 avg_val_loss:0.6858,val_auc:0.9429,lr:0.00100000
epoch:5 avg_loss:0.6052 avg_val_loss:0.6715,val_auc:0.9845,lr:0.00100000
epoch:6 avg_loss:0.5957 avg_val_loss:0.6510,val_auc:0.9886,lr:0.00100000
epoch:7 avg_loss:0.5709 avg_val_loss:0.6211,val_auc:0.9935,lr:0.00100000
epoch:8 avg_loss:0.5982 avg_val_loss:0.5851,val_auc:0.9951,lr:0.00100000
epoch:9 avg_loss:0.5814 avg_val_loss:0.5467,val_auc:0.9959,lr:0.00100000
epoch:10 avg_loss:0.5208 avg_val_loss:0.5103,val_auc:0.9967,lr:0.00100000
epoch:11 avg_loss:0.5548 avg_val_loss:0.4737,val_auc:0.9976,lr:0.00100000
epoch:12 avg_loss:0.4676 avg_val_loss:0.4446,val_auc:0.9984,lr:0.00100000
epoch:13 avg_loss:0.4221 avg_val_loss:0.4134,val_auc:0.9992,lr:0.00100000
epoch:14 avg_loss: